---
title: Глава 11. Условная логика
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-10-11
abbreviations:
    MyST: Markedly Structured Text
    ПКМ: Правая Кнопка Мыши
    Jupyter Book: Инструмент сборки статических сайтов
    JupySQL: Расширение для запуска и подсветки SQL в Jupyter
    GitHub: Платформа хостинга репозиториев и совместной разработки
    GitHub Pages: Сервис бесплатного хостинга статических сайтов
    GitHub Actions: Платформа автоматизации рабочих процессов и CI/CD
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

In [1]:
import pandas as pd
import pymysql
import sqlalchemy as sa
import sql

pd.set_option('display.max_rows', 20)

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.autopandas = True

connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
engine = sa.create_engine(connection_url)

%sql engine

print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"PyMySQL ver. {pymysql.__version__}: DBAPI-драйвер подключения")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
PyMySQL ver. 2.2.8: DBAPI-драйвер подключения
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


В некоторых ситуациях требуется, чтобы SQL-логика разветвлялась в том или ином направлении в зависимости от значений определенных столбцов или выражений. Из этой главы вы узнаете, как писать операторы, которые могут вести себя по-разному в зависимости от данных, полученных во время выполнения инструкции. Механизм условной логики в инструкциях SQL – это выражение `case`, которое можно использовать в инструкциях `select`, `insert`, `update` и `delete`.

## Что такое условная логика

Условная логика – это просто возможность выбрать один из нескольких путей во время выполнения программы. Например, при запросе информации о клиенте вы можете захотеть включить столбец **customer.active**, в котором хранится **1** для обозначения активного и **0** – для обозначения неактивного клиента. Если результаты запроса используются для создания отчета, то вы можете захотеть перевести это значение в строку, чтобы улучшить удобочитаемость.

Хотя каждая база данных включает встроенные функции для подобных ситуаций, общих стандартов нет, так что нужно запоминать в какой базе данных какие функции имеются. К счастью, каждая реализация базы данных SQL включает выражение `case`, которое полезно во многих ситуациях, в том числе и для простого перевода:

:::{attention}
:class: dropdown
:open: false

В запросе автора элементы в инструкции `SELECT` расположены таким образом, что может **ошибочно** показаться как будто список полей состоит только из двух элементов **first_name, last_name**, а дальше с новой строки пошла какая-то новая крутая конструкция `CASE`, значит после **last_name** запятая не нужна.

Но колонка **last_name** НЕ последняя! \
Последней колонкой является **activity_type**, вычисленная через `CASE`.

```sql
SELECT
    first_name,          -- колонка 1 (нужна запятая)
    last_name,           -- колонка 2 (нужна запятая)
    CASE                 -- колонка 3 (НАЧАЛАСЬ)
        WHEN active = 1 THEN 'ACTIVE'
        ELSE 'INACTIVE'
    END AS activity_type -- колонка 3 (ЗАКОНЧИЛАСЬ, запятой нет)
FROM customer;
```

Вся конструкция `CASE ... END` – это **не** отдельная секция SQL вроде `FROM` или `WHERE`, это всего лишь вычисляемая колонка (такая же как `COUNT(*)` или `amount*2`).
:::

In [3]:
%%sql
SELECT first_name, last_name,
    CASE
        WHEN active = 1 THEN 'ACTIVE'
        ELSE 'INACTIVE'
    END activity_type
FROM customer;

599 rows affected.

,first_name,last_name,activity_type
0,MARY,SMITH,ACTIVE
1,PATRICIA,JOHNSON,ACTIVE
2,LINDA,WILLIAMS,ACTIVE
3,BARBARA,JONES,ACTIVE
4,ELIZABETH,BROWN,ACTIVE
...,...,...,...
594,TERRENCE,GUNDERSON,ACTIVE
595,ENRIQUE,FORSYTHE,ACTIVE
596,FREDDIE,DUGGAN,ACTIVE
597,WADE,DELVALLE,ACTIVE


Этот запрос включает выражение `case` для генерации значения столбца **activity_type**, которое возвращает строку ACTIVE или INACTIVE в зависимости от значения столбца **customer.active**.

## Выражение *case*

Все основные серверы баз данных включают встроенные функции, имитирующие инструкцию **if-then-else**, имеющуюся в большинстве языков программирования (например, в Oracle – функция `decode()`, в MySQL – `if()`, в SQL Server – `coalesce()`).

Выражения `case` также предназначены для облегчения применения логики **if-then-else**, но при этом имеют следующие преимущества перед встроенными функциями.
- Выражение `case` является частью стандарта SQL (выпуск SQL92) и реализовано в Oracle Database, SQL Server, MySQL, PostgreSQL, IBM UDB и других СУБД.
- Выражения `case` встроены в грамматику SQL и могут быть включены в инструменты `select`, `insert`, `update` и `delete`.

В следующих двух подразделах представлены два разных типа выражений `case`. Материал сопровождается некоторыми примерами case-выражений в действии.

### Поисковые выражения case

Выражение `case`, показанное ранее в этой главе, является примером поискового выражения `case`, имеющего следующий синтаксис:

```sql
CASE
    WHEN C1 THEN E1
    WHEN C2 THEN E2
    ...
    WHEN CN THEN EN
    [ELSE ED]
END
```

В этом определении символы C1, C2,... CN представляют условия (Condition), а символы E1, E2,... EN – выражения (Expression), возвращаемые выражением `case`. Если условие в предложении `when` вычисляется как истинное, выражение `case` возвращает соответствующее выражение E\#.

Кроме того, символ ED представляет выражение по умолчанию (Expression Default), которое выражение `case` возвращает, если *ни одно* из условий C1, C2,... CN не дает при вычислении `true`. Предложение `else` является необязательным, поэтому оно заключено в квадратные скобки.

Все выражения, возвращаемые различными предложениями `when` должны иметь один и тот же тип (например, `date`, `number`, `varchar`).

Вот пример поискового выражения `case`:

```sql
CASE
    WHEN category.name IN ('Children', 'Family', 'Sports', 'Animation')
        THEN 'All Ages'
    WHEN category.name = 'Horror'
        THEN 'Adult'
    WHEN category.name IN ('Music', 'Games')
        THEN 'Teens'
    ELSE 'Other'
END
```

Это выражение `case` возвращает строку, которую можно использовать для классификации фильмов в зависимости от их категории. При обработке выражения `case` по порядку сверху вних вычисляются предложения `when`. Как только одно из условий в предложении `when` дает значение `true`, возвращается соответствующее выражение, а все оставшиеся предложения `when` игнорируются. Если ни одно из условий в предложениях `when` не является истинным, возвращается выражение в предложении `else`.

Хотя предыдущий пример возвращает строковые выражения, имейте в виду, что выражения `case` могут возвращать выражения любого типа, включая подзапросы. Вот еще одна версия показанного выше запроса, в которой для возврата количества прокатов **но только для активных клиентов** используется подзапрос:

:::{note} Оригинал автора
:class: simple dropdown
:icon: false
:open: false
Автору приходится использовать *типографские приемы*, которые в реальной разработке никогда не применяются.
```sql
SELECT c.first_name, c.last_name,
  CASE
    WHEN active = 0 THEN 0
    ELSE
     (SELECT COUNT(*) FROM rental r
      WHERE r.customer_id = c.customer_id)
  END num_rentals
FROM customer c;
```
:::

In [2]:
%%sql
SELECT c.first_name, c.last_name,
    CASE
        WHEN active = 0 THEN 0
        ELSE (
            SELECT COUNT(*)
            FROM rental r
            WHERE r.customer_id = c.customer_id
        )
    END num_rentals
FROM customer c;

599 rows affected.

,first_name,last_name,num_rentals
0,MARY,SMITH,32
1,PATRICIA,JOHNSON,27
2,LINDA,WILLIAMS,26
3,BARBARA,JONES,22
4,ELIZABETH,BROWN,38
...,...,...,...
594,TERRENCE,GUNDERSON,30
595,ENRIQUE,FORSYTHE,28
596,FREDDIE,DUGGAN,25
597,WADE,DELVALLE,22


Эта версия запроса использует коррелированный подзапрос для получения количества прокатов для каждого активного клиента. В зависимости от процента активных клиентов использование этого подхода может быть более эффективным, чем соединение таблиц **customer** и **rental** и группировка по столбцу **customer_id**.

### Простые выражения case

Простое выражение `case` очень похоже на поисковое выражение `case`, но является немного менее гибким. Вот его синтаксис:

```sql
CASE V0
    WHEN V1 THEN E1
    WHEN V2 THEN E2
    ...
    WHEN VN THEN EN
    [ELSE ED]
END
```
В этом определинии V0 представляет значение, а символы V1, V2,... VN – значения, с которыми необходимо сравнивать V0. Символы E1, E2,... EN обозначают выражения, которые должны быть возвращены выражением `case`, а ED – выражение, которое должно быть возвращено, если ни одно из значений в наборе V1, V2,... VN не созпадает со значением V0.

Вот пример простого выражения `case`:

```sql
CASE category.name
    WHEN 'Children' THEN 'All Ages'
    WHEN 'Family' THEN 'All Ages'
    WHEN 'Sports' THEN 'All Ages'
    WHEN 'Animation' THEN 'All Ages'
    WHEN 'Horror' THEN 'Adult'
    WHEN 'Music' THEN 'Teens'
    WHEN 'Games' THEN 'Teens'
    ELSE 'Other'
END
```

Простые выражения `case` менее гибки, чем поисковые, потому что вы не можете указывать собственные условия, тогда как поисковые выражения `case` могут включать условия диапазона, условия неравенства и многокомпонетные условия с использованием **and/or/not**. Поэтому я рекомендую использовать поисковые выражения `case` везде, кроме простейших случаев.

## Примеры выражений case

В следующих разделах представлены различные примеры, иллюстрирующие полезность условной логики в инструкциях SQL.

### Преобразования результирующего набора

Возможно, вы столкнулись с ситуацией, когда вы выполняете агрегирование конечного набора значений, таких как дни недели, но хотите чтобы результирующий набор содержал единственную строку со столбцами, по одному для каждого значения, а не по одной строке для каждого значения.

В качестве примера предположим, что требуется написать запрос, который показывает количество прокатов фильмов в мае, июне и июле 2005 года:

In [3]:
%config SqlMagic.autopandas = False

In [7]:
%%sql
SELECT
    MONTHNAME(rental_date) AS rental_month,
    COUNT(*) AS num_rentals
FROM rental
WHERE rental_date BETWEEN '2005-05-01' AND '2005-08-01'
GROUP BY MONTHNAME(rental_date);

3 rows affected.

rental_month,num_rentals
May,1156
June,2311
July,6709


Однако на запрос накладывается условие – он должен вернуть одну строку данных с тремя столбцами (по одному для каждого из трех месяцев). Чтобы преобразовать этот результирующий набор в единую строку, нужно создать три столбца и в каждом столбце суммировать *только* те строки, окторые относятся к рассматриваемому месяцу:

In [12]:
%%sql
SELECT
    SUM(CASE WHEN MONTHNAME(rental_date) = 'May' THEN 1
             ELSE 0 END) May_rentals,
    SUM(CASE WHEN MONTHNAME(rental_date) = 'June' THEN 1
             ELSE 0 END) June_rentals,
    SUM(CASE WHEN MONTHNAME(rental_date) = 'July' THEN 1
             ELSE 0 END) July_rentals
FROM rental
WHERE rental_date BETWEEN '2005-05-01' AND '2005-08-01';

1 rows affected.

May_rentals,June_rentals,July_rentals
1156,2311,6709


Все три столбца в предыдущем запросе идентичны, за исключением значения месяца. Когда функция `monthname()` возвращает требуемое значение для конкретного столбца, выражение `case` возвращает значение 1; в противном случае возвращается 0. При суммировании по всем строкам каждый столбец возвращает количество счетов, открытых в этом месяце. Очевидно. что такие преобразования применимы только для небольшого количества значений. Генерация по одному столбцу для каждого года, начиная с 1905 года, быстро станет утомительной.

:::{note}
Хотя это и несколько выходит за рамки данной книги, стоит отметить, что и SQL Server и Oracle Database включают предложения `pivot` специально для такого рода запросов.

### Проверка существования

Иногда нужно определить, существуют ли взаимосвязи между двумя сущностями без учета количества. Например, вы можете захотеть узнать, снялся ли некоторый актер хотя бы в одном фильме с рейтингом G, без учета фактического количества фильмов.

Вот запрос для генерации трех выходных столбцов, один из которых показывает, снимался ли актер в фильмах с рейтингом G, еще один – для фильмов с рейтингом PG и третий – для фильмов с рейтингом NC-17:

In [14]:
%config SqlMagic.displaylimit = 25

:::{note} Оригинал автора
:class: simple dropdown
:icon: false
:open: false
Автору приходится использовать *типографские приемы*, которые в реальной разработке никогда не применяются.
```sql
%%sql
SELECT
    a.first_name,
    a.last_name,
    CASE
        WHEN EXISTS (SELECT 1 FROM film_actor fa
                      INNER JOIN film f ON fa.film_id = f.film_id
                      WHERE fa.actor_id = a.actor_id
                        AND f.rating = 'G') THEN 'Y'
        ELSE 'N'
    END g_actor,
    CASE
        WHEN EXISTS (SELECT 1 FROM film_actor fa
                      INNER JOIN film f ON fa.film_id = f.film_id
                      WHERE fa.actor_id = a.actor_id
                        AND f.rating = 'PG') THEN 'Y'
        ELSE 'N'
    END pg_actor,
    CASE
        WHEN EXISTS (SELECT 1 FROM film_actor fa
                      INNER JOIN film f ON fa.film_id = f.film_id
                      WHERE fa.actor_id = a.actor_id
                        AND f.rating = 'NC-17') THEN 'Y'
        ELSE 'N'
    END nc17_actor
FROM actor a
WHERE a.last_name LIKE 'S%' OR a.first_name LIKE 'S%';
```
:::

In [ ]:
%%sql
SELECT
    a.first_name,
    a.last_name,

    CASE
        WHEN EXISTS (
            SELECT 1
            FROM film_actor fa
                INNER JOIN film f ON fa.film_id = f.film_id
            WHERE fa.actor_id = a.actor_id
                AND f.rating = 'G'
        ) THEN 'Y'
        ELSE 'N'
    END AS g_actor,

    CASE
        WHEN EXISTS (
            SELECT 1
            FROM film_actor fa
                INNER JOIN film f ON fa.film_id = f.film_id
            WHERE fa.actor_id = a.actor_id
                AND f.rating = 'PG'
        ) THEN 'Y'
        ELSE 'N'
    END AS pg_actor,

    CASE
        WHEN EXISTS (
            SELECT 1
            FROM film_actor fa
                INNER JOIN film f ON fa.film_id = f.film_id
            WHERE fa.actor_id = a.actor_id
                AND f.rating = 'NC-17'
        ) THEN 'Y'
        ELSE 'N'
    END AS nc17_actor

FROM actor a
WHERE a.last_name LIKE 'S%'
    OR a.first_name LIKE 'S%';

22 rows affected.

first_name,last_name,g_actor,pg_actor,nc17_actor
JOE,SWANK,Y,Y,Y
SANDRA,KILMER,Y,Y,Y
CAMERON,STREEP,Y,Y,Y
SANDRA,PECK,Y,Y,Y
SISSY,SOBIESKI,Y,Y,N
NICK,STALLONE,Y,Y,Y
SEAN,WILLIAMS,Y,Y,Y
GROUCHO,SINATRA,Y,Y,Y
SCARLETT,DAMON,Y,Y,Y
SPENCER,PECK,Y,Y,Y


Каждое выражение `case` включает коррелированный подзапрос к таблицам **film_actor** и **film**. Один из них ищет фильмы с рейтингом G, второй – фильмы с рейтингом PG и третий – фильмы с рейтингом NC-17. Поскольку в каждом предложении `when` используется оператор `exists`, условия вычисляются как истинные, если актер появился как минимум в одном фильме с соответствующим рейтингом.

В других случаях может быть интересно, сколько именно строк имеется, но только до определенного предела. Например, в следующем запросе простое выражение `case` используется для подсчета количества копий в прокате для каждого фильма, возвращая строки 'Out Of Stok' (нет в прокате), 'Scarce' (редкий), 'Available' (доступный) или 'Common' (обычный, в большом количестве):

In [5]:
%config SqlMagic.autopandas = True

In [2]:
%%sql
SELECT
    f.title,
    CASE (
        SELECT COUNT(*)
        FROM inventory i
        WHERE i.film_id = f.film_id
    )
        WHEN 0 THEN 'Out Of Stock'
        WHEN 1 THEN 'Scarce'
        WHEN 2 THEN 'Scarce'
        WHEN 3 THEN 'Available'
        WHEN 4 THEN 'Available'
        ELSE 'Common'
    END AS film_availability
FROM film f;

1000 rows affected.

,title,film_availability
0,ACADEMY DINOSAUR,Common
1,ACE GOLDFINGER,Available
2,ADAPTATION HOLES,Available
3,AFFAIR PREJUDICE,Common
4,AFRICAN EGG,Available
...,...,...
995,YOUNG LANGUAGE,Scarce
996,YOUTH KICK,Scarce
997,ZHIVAGO CORE,Scarce
998,ZOOLANDER FICTION,Common


В этом запросе я перестал вести подсчет количества после 5, присваивая любому другому числу больше 5 метку 'Common'.

### Ошибки деления на нуль

При выполнении вычислений, включающих деление, всегда следует озаботиться тем, чтобы знаменатели никогда не были равны нулю. В то время как некоторые серверы баз данных, такие как Oracle Database, сообщат об ошибке, MySQL, встретив нулевой знаменатель, просто установит результат вычисления равным `null`:

In [4]:
%%sql
SELECT 100 / 0;

1 rows affected.

100 / 0
None


Чтобы защитить свои расчеты от ошибок или, что еще хуже, от таинственным образом появляющихся значений `null`, следует *завернуть* все знаменатели в условную логику, как демонстрируется в следующем примере:

:::{note} Оригинал автора
:class: simple dropdown
:icon: false
:open: false

```sql
SELECT c.first_name, c.last_name,
  SUM(p.amount) tot_payment_amt,
  COUNT(p.amount) num_payments,
  SUM(p.amount) /
    CASE WHEN COUNT(p.amount) = 0 THEN 1
         ELSE COUNT(p.amount)
    END avg_payment
FROM customer c
  LEFT OUTER JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.first_name, c.last_name;
```
:::

In [6]:
%%sql
SELECT
    c.first_name,
    c.last_name,
    SUM(p.amount) AS tot_payment_amt,
    COUNT(p.amount) AS num_payments,
    SUM(p.amount) / CASE
        WHEN COUNT(p.amount) = 0 THEN 1
        ELSE COUNT(p.amount)
    END AS avg_payment
FROM customer c
    LEFT JOIN payment p
        ON c.customer_id = p.customer_id
GROUP BY
    c.first_name,
    c.last_name;

599 rows affected.

,first_name,last_name,tot_payment_amt,num_payments,avg_payment
0,MARY,SMITH,118.68,32,3.708750
1,PATRICIA,JOHNSON,128.73,27,4.767778
2,LINDA,WILLIAMS,135.74,26,5.220769
3,BARBARA,JONES,81.78,22,3.717273
4,ELIZABETH,BROWN,144.62,38,3.805789
...,...,...,...,...,...
594,TERRENCE,GUNDERSON,117.70,30,3.923333
595,ENRIQUE,FORSYTHE,96.72,28,3.454286
596,FREDDIE,DUGGAN,99.75,25,3.990000
597,WADE,DELVALLE,83.78,22,3.808182


Этот запрос вычисляет среднюю сумму платежа для каждого клиента. Поскольку некоторые клиенты могут быть новыми и еще не брали напрокат ни одного фильма, лучше включить выражение `case`, чтобы знаменатель никогда не был равен нулю.